In [ ]:
"""Every trained projection against the floor it had to clear.

Sandbox. Not committed. Only the PNG under assets/ is.

The y axis is loss divided by the null, so the floor is the line at one and
the pass criterion is a position on the page. That also makes keys and values
comparable: their targets differ in scale by orders of magnitude across
depth, and a raw loss axis would show that difference and nothing else.

Whether this figure should move was left to be decided after the run. It
should not, though not for the reason first written here. Five curves do
cross the line, all of them at epoch zero and all of them downward: an
untrained projection sits at the null, which is what the null means. No curve
ends above it and none returns to it. An animation would show every line
falling below the threshold in its first frame and then holding, which is a
sequence with one event in it and that event is the initialisation.
"""

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

LOG_PATH = Path("results") / "train_log.json"
ASSETS = Path("assets")

PRIMARY = "#5C1D74"
SECONDARY = "#E5C300"
TERTIARY = "#1D7466"
TEXT_COLOR = "#25172A"
BG_COLOR = "#FCFAFF"
MUTED = "#B9AEC2"
GOLD_INK = "#8A7600"

# A two way categorical is violet against gold. Teal is the reference line,
# which is a role of a different kind rather than a third series.
KEYS_COLOR = PRIMARY
VALUES_COLOR = SECONDARY
NULL_COLOR = TERTIARY

DPI = 200


def load():
    if not LOG_PATH.exists():
        raise SystemExit(f"missing {LOG_PATH}; run scripts/train_projection.py first")
    return json.loads(LOG_PATH.read_text(encoding="utf-8"))


def build(log):
    fig, ax = plt.subplots(figsize=(9.6, 5.8), facecolor=BG_COLOR)
    fig.subplots_adjust(top=0.84, bottom=0.13, left=0.10, right=0.97)
    ax.set_facecolor(BG_COLOR)

    ax.axhline(1.0, color=NULL_COLOR, linewidth=1.8, linestyle=(0, (5, 3)), zorder=4)
    ax.annotate(
        "the null point",
        xy=(0.985, 1.012), xycoords=("axes fraction", "data"),
        ha="right", va="bottom", fontsize=9.5, color=NULL_COLOR, zorder=6,
    )

    highest = 0.0
    for record in log["per_layer"]:
        kind = record["kind"]
        colour = KEYS_COLOR if kind == "keys" else VALUES_COLOR
        null = record["null_held_out"]
        epochs = [p["epoch"] for p in record["curve"]]
        losses = [p["held_out"] / null for p in record["curve"]]
        highest = max(highest, max(losses))
        ax.plot(epochs, losses, color=colour, linewidth=0.85, alpha=0.42,
                zorder=2 if kind == "keys" else 3)
        ax.plot([record["selected_epoch"]], [record["relative_held_out"]],
                marker="o", markersize=3.4, color=colour,
                markeredgecolor=TEXT_COLOR, markeredgewidth=0.4, zorder=5)

    ax.set_xlabel("training epoch", fontsize=10, fontweight="bold", color=TEXT_COLOR)
    ax.set_ylabel("held out loss", fontsize=10, fontweight="bold", color=TEXT_COLOR)
    ax.set_ylim(0.25, max(1.12, highest * 1.04))
    ax.set_xlim(-4, 204)
    ax.grid(True, color=MUTED, alpha=0.22, linewidth=0.6)
    ax.tick_params(colors=TEXT_COLOR, labelsize=9)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(MUTED)

    n = log["summary"]["keys"]["n_layers"]
    ax.set_title(
        f"held out loss relative to the null, {n} paired layers, keys and values "
        f"lower in all {n} layers",
        fontsize=11.5, color=TEXT_COLOR, pad=26, fontweight="bold"
    )
    ax.annotate(
        "one line per layer, dot at the epoch chosen on the validation split",
        xy=(0.5, 1.015), xycoords="axes fraction", ha="center", va="bottom",
        fontsize=9.2, color="#6E6478",
    )

    legend = ax.legend(
        handles=[
            Line2D([], [], color=KEYS_COLOR, linewidth=2.0, label="keys"),
            Line2D([], [], color=VALUES_COLOR, linewidth=2.4, label="values"),
        ],
        loc="lower left", frameon=False, fontsize=10.5,
    )
    for text in legend.get_texts():
        text.set_color(TEXT_COLOR)

    ASSETS.mkdir(parents=True, exist_ok=True)
    out = ASSETS / "c2c_loss_vs_null.png"
    fig.savefig(out, dpi=DPI, facecolor=BG_COLOR)
    return fig, ax, out, highest


def main():
    log = load()
    fig, ax, out, highest = build(log)
    plt.close(fig)
    print(out, f"| highest point on any curve: {highest:.4f}")


if __name__ == "__main__":
    main()

assets/c2c_loss_vs_null.png | highest point on any curve: 1.0818
